# SwinJSCC — Kaggle 2×T4 DataParallel Training

This notebook trains the official high-resolution **SwinJSCC Base** architecture with:

- `SwinJSCC_w/o_SAandRA` pretraining
- `SwinJSCC_w/_SAandRA` full model
- Channel ModNet
- Rate ModNet
- AWGN channel
- MSE reconstruction loss
- PyTorch `DataParallel` across both Kaggle T4 GPUs

For 256×256 input, the official Base configuration produces a final Swin feature with 320 channels at 16×16 spatial resolution:

\[
Z\in R^{B	imes256	imes320}.
\]

The notebook also evaluates PSNR versus SNR/rate and reports the actual SwinJSCC CBR and latent representation sizes.


## Dataset: exactly what is being used

**Default training source in this notebook:**

```text
/kaggle/input/datasets/sherylmehta/kodak-dataset
```

Because this is a small Kodak collection, the notebook splits the images at the **image level**:

- 80% → training images
- 20% → held-out test images

Training images are randomly cropped to 256×256 and augmented.

This is therefore a **Kodak development experiment**, not the original paper training protocol.

The official SwinJSCC repository uses:

```text
DIV2K → training
Kodak → testing
```

for its high-resolution experiments and recommends first training `SwinJSCC_w/o_SAandRA`, then using it to initialize the complete SA+RA model.

If you later add DIV2K to Kaggle, set `DIV2K_DIR` in the configuration cell and the notebook can use it as the training set.

Do not describe the Kodak-only result as an exact reproduction of the paper.


In [ ]:
import os, sys, math, random, time, subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))


In [ ]:
KAGGLE_KODAK_DIR = Path(
    "/kaggle/input/datasets/sherylmehta/kodak-dataset"
)

# Optional future DIV2K location.
# Example:
# DIV2K_DIR = Path("/kaggle/input/div2k/DIV2K_train_HR")
DIV2K_DIR = None

IMAGE_SIZE = 256
USE_DIV2K = DIV2K_DIR is not None and Path(DIV2K_DIR).exists()

TRAIN_DIR = Path(DIV2K_DIR) if USE_DIV2K else KAGGLE_KODAK_DIR

if not TRAIN_DIR.exists():
    raise FileNotFoundError(TRAIN_DIR)

files = sorted([
    p for p in TRAIN_DIR.rglob("*")
    if p.suffix.lower() in {
        ".png", ".jpg", ".jpeg", ".bmp", ".webp", ".tif", ".tiff"
    }
])

print("Training dataset:", TRAIN_DIR)
print("DIV2K mode:", USE_DIV2K)
print("Images found:", len(files))


In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

random.shuffle(files)

split = max(1, int(0.80 * len(files)))
train_files = files[:split]
test_files = files[split:]

print("Training images:", len(train_files))
print("Held-out test images:", len(test_files))


In [ ]:
class KodakPatchDataset(Dataset):
    def __init__(self, files, train=True, size=256):
        self.files = list(files)
        self.train = train
        self.size = size

        if train:
            self.transform = transforms.Compose([
                transforms.RandomCrop(size),
                transforms.RandomHorizontalFlip(),
                transforms.RandomVerticalFlip(),
                transforms.ToTensor(),
            ])
        else:
            self.transform = transforms.Compose([
                transforms.CenterCrop(size),
                transforms.ToTensor(),
            ])

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        path = self.files[idx]

        with Image.open(path) as im:
            im = im.convert("RGB")

            if im.width < self.size or im.height < self.size:
                scale = max(
                    self.size / im.width,
                    self.size / im.height
                )
                new_size = (
                    math.ceil(im.width * scale),
                    math.ceil(im.height * scale)
                )
                im = im.resize(new_size, Image.Resampling.BICUBIC)

            return self.transform(im)


train_ds = KodakPatchDataset(train_files, True, IMAGE_SIZE)
test_ds = KodakPatchDataset(test_files, False, IMAGE_SIZE)

BATCH_SIZE = 8
NUM_WORKERS = min(4, os.cpu_count() or 1)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=NUM_WORKERS > 0,
)

test_loader = DataLoader(
    test_ds,
    batch_size=4,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=NUM_WORKERS > 0,
)

print("Train batches:", len(train_loader))
print("Test batches:", len(test_loader))


## Official SwinJSCC source

The notebook clones the official `semcomm/SwinJSCC` repository instead of implementing a different SwinJSCC from scratch.

The official high-resolution Base configuration is:

```text
embed_dims = [128, 192, 256, 320]
depths     = [2, 2, 6, 2]
heads      = [4, 6, 8, 10]
patch_size = 2
window     = 8
```

The official repository states that HR training should first pretrain the model without SA/RA and then train the complete model.

One implementation detail matters for our two-GPU setup: the official `SwinJSCC.forward()` returns Python scalar metrics along with tensors. `nn.DataParallel` cannot reliably gather that structure. Therefore this notebook uses the official encoder, decoder and channel classes inside a wrapper whose forward returns tensors only.


In [ ]:
REPO_DIR = Path("/kaggle/working/SwinJSCC_official")

if not REPO_DIR.exists():
    subprocess.run([
        "git", "clone", "--depth", "1",
        "https://github.com/semcomm/SwinJSCC.git",
        str(REPO_DIR)
    ], check=True)

sys.path.insert(0, str(REPO_DIR))

from net.encoder import create_encoder
from net.decoder import create_decoder
from net.channel import Channel

print("Official SwinJSCC imported.")


In [ ]:
MODEL_BASE = "SwinJSCC_w/o_SAandRA"
MODEL_FULL = "SwinJSCC_w/_SAandRA"

EMBED_DIMS = [128, 192, 256, 320]
DEPTHS = [2, 2, 6, 2]
NUM_HEADS = [4, 6, 8, 10]

PATCH_SIZE = 2
WINDOW_SIZE = 8
MLP_RATIO = 4.0

PRETRAIN_SNR = 10
PRETRAIN_C = 96

TRAIN_SNRS = [1, 4, 7, 10, 13]
TRAIN_RATES = [32, 64, 96, 128, 192]

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

def kwargs_for(model, C):
    return dict(
        model=model,
        img_size=(IMAGE_SIZE, IMAGE_SIZE),
        patch_size=PATCH_SIZE,
        in_chans=3,
        embed_dims=EMBED_DIMS,
        depths=DEPTHS,
        num_heads=NUM_HEADS,
        C=C,
        window_size=WINDOW_SIZE,
        mlp_ratio=MLP_RATIO,
        qkv_bias=True,
        qk_scale=None,
        norm_layer=nn.LayerNorm,
        patch_norm=True,
    )

enc_base = kwargs_for(MODEL_BASE, PRETRAIN_C)
dec_base = dict(
    model=MODEL_BASE,
    img_size=(IMAGE_SIZE, IMAGE_SIZE),
    embed_dims=EMBED_DIMS[::-1],
    depths=DEPTHS[::-1],
    num_heads=NUM_HEADS[::-1],
    C=PRETRAIN_C,
    window_size=WINDOW_SIZE,
    mlp_ratio=MLP_RATIO,
    qkv_bias=True,
    qk_scale=None,
    norm_layer=nn.LayerNorm,
    patch_norm=True,
)

enc_full = kwargs_for(MODEL_FULL, None)
dec_full = dict(
    model=MODEL_FULL,
    img_size=(IMAGE_SIZE, IMAGE_SIZE),
    embed_dims=EMBED_DIMS[::-1],
    depths=DEPTHS[::-1],
    num_heads=NUM_HEADS[::-1],
    C=None,
    window_size=WINDOW_SIZE,
    mlp_ratio=MLP_RATIO,
    qkv_bias=True,
    qk_scale=None,
    norm_layer=nn.LayerNorm,
    patch_norm=True,
)


In [ ]:
class TrainableSwinJSCC(nn.Module):
    def __init__(self, model_name, encoder_kwargs, decoder_kwargs):
        super().__init__()

        self.model_name = model_name
        self.encoder = create_encoder(**encoder_kwargs)
        self.decoder = create_decoder(**decoder_kwargs)

        class Args:
            pass

        class Config:
            pass

        args = Args()
        config = Config()

        args.channel_type = "awgn"
        config.pass_channel = True

        self.channel = Channel(args, config)

        self.H = 0
        self.W = 0
        self.downsample = 4

    def update_resolution(self, H, W):
        if H != self.H or W != self.W:
            self.encoder.update_resolution(H, W)
            self.decoder.update_resolution(
                H // (2 ** self.downsample),
                W // (2 ** self.downsample)
            )
            self.H = H
            self.W = W

    def forward(self, images, snr, rate):
        B, _, H, W = images.shape
        self.update_resolution(H, W)

        encoded = self.encoder(
            images,
            snr,
            rate,
            self.model_name
        )

        if self.model_name in {
            "SwinJSCC_w/_RA",
            "SwinJSCC_w/_SAandRA"
        }:
            feature, mask = encoded

            avg_power = (
                torch.sum(feature ** 2) /
                mask.sum().clamp_min(1.0)
            )

            noisy = self.channel.forward(
                feature,
                snr,
                avg_power
            )

            noisy = noisy * mask

        else:
            feature = encoded

            avg_power = torch.mean(
                feature ** 2
            ).clamp_min(1e-12)

            noisy = self.channel.forward(
                feature,
                snr,
                avg_power
            )

            mask = torch.ones_like(feature)

        reconstruction = self.decoder(
            noisy,
            snr,
            self.model_name
        ).clamp(0, 1)

        # DataParallel-safe: tensors only.
        return reconstruction, feature, mask


def parallel(model):
    model = model.to(DEVICE)

    if torch.cuda.device_count() >= 2:
        model = nn.DataParallel(
            model,
            device_ids=list(range(torch.cuda.device_count())),
            output_device=0
        )
        print("DataParallel devices:", model.device_ids)
    else:
        print("Single GPU.")

    return model


baseline_model = parallel(
    TrainableSwinJSCC(
        MODEL_BASE,
        enc_base,
        dec_base
    )
)

full_model = parallel(
    TrainableSwinJSCC(
        MODEL_FULL,
        enc_full,
        dec_full
    )
)

print(
    "Baseline parameters:",
    sum(p.numel() for p in baseline_model.parameters()) / 1e6,
    "M"
)

print(
    "SA+RA parameters:",
    sum(p.numel() for p in full_model.parameters()) / 1e6,
    "M"
)


In [ ]:
# Mandatory forward test before training.

sample = next(iter(train_loader))[:8].to(DEVICE)

baseline_model.eval()
full_model.eval()

with torch.no_grad():
    r0, z0, m0 = baseline_model(
        sample, PRETRAIN_SNR, PRETRAIN_C
    )

    r1, z1, m1 = full_model(
        sample, 10, 96
    )

print("Baseline reconstruction:", r0.shape)
print("Baseline latent:", z0.shape)

print("SA+RA reconstruction:", r1.shape)
print("SA+RA latent:", z1.shape)
print("SA+RA mask:", m1.shape)

assert r1.shape == sample.shape
assert z1.ndim == 3
assert m1.shape == z1.shape
assert z1.shape[-1] == 320

print("Forward test PASSED.")


# Stage 1 — fixed-rate pretraining

Train `SwinJSCC_w/o_SAandRA` at:

\[
SNR=10,\quad C=96.
\]

This follows the official high-resolution training strategy.


In [ ]:
PRETRAIN_EPOCHS = 5
LR = 2e-4

opt = torch.optim.AdamW(
    baseline_model.parameters(),
    lr=LR,
    weight_decay=1e-4
)

scaler = torch.cuda.amp.GradScaler(
    enabled=torch.cuda.is_available()
)

pretrain_history = []

for epoch in range(PRETRAIN_EPOCHS):
    baseline_model.train()

    total = 0.0
    count = 0

    t0 = time.time()

    for images in train_loader:
        images = images.to(DEVICE, non_blocking=True)

        opt.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(
            enabled=torch.cuda.is_available()
        ):
            recon, _, _ = baseline_model(
                images,
                PRETRAIN_SNR,
                PRETRAIN_C
            )

            loss = F.mse_loss(recon, images)

        scaler.scale(loss).backward()
        scaler.unscale_(opt)

        torch.nn.utils.clip_grad_norm_(
            baseline_model.parameters(), 1.0
        )

        scaler.step(opt)
        scaler.update()

        total += loss.item() * images.size(0)
        count += images.size(0)

    epoch_loss = total / count
    pretrain_history.append(epoch_loss)

    print(
        f"Pretrain {epoch+1}/{PRETRAIN_EPOCHS}: "
        f"MSE={epoch_loss:.6f}, "
        f"time={time.time()-t0:.1f}s"
    )


# Stage 2 — full SA+RA training

The full model uses:

\[
SNR\in\{1,4,7,10,13\}
\]

and

\[
C\in\{32,64,96,128,192\}.
\]

The final bottleneck still has 320 feature channels. The Rate ModNet creates a mask selecting the requested rate.


In [ ]:
# Transfer compatible backbone weights.

base = (
    baseline_model.module
    if isinstance(baseline_model, nn.DataParallel)
    else baseline_model
)

full = (
    full_model.module
    if isinstance(full_model, nn.DataParallel)
    else full_model
)

base_state = base.state_dict()
full_state = full.state_dict()

compatible = {
    k: v
    for k, v in base_state.items()
    if k in full_state and full_state[k].shape == v.shape
}

full_state.update(compatible)
full.load_state_dict(full_state, strict=False)

print(
    "Transferred:",
    len(compatible),
    "compatible tensors"
)


In [ ]:
FULL_EPOCHS = 5
FULL_LR = 1e-4

opt_full = torch.optim.AdamW(
    full_model.parameters(),
    lr=FULL_LR,
    weight_decay=1e-4
)

scaler_full = torch.cuda.amp.GradScaler(
    enabled=torch.cuda.is_available()
)

full_history = []

for epoch in range(FULL_EPOCHS):
    full_model.train()

    total = 0.0
    count = 0
    t0 = time.time()

    for images in train_loader:
        images = images.to(DEVICE, non_blocking=True)

        snr = random.choice(TRAIN_SNRS)
        rate = random.choice(TRAIN_RATES)

        opt_full.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(
            enabled=torch.cuda.is_available()
        ):
            recon, _, _ = full_model(
                images,
                snr,
                rate
            )

            loss = F.mse_loss(
                recon, images
            )

        scaler_full.scale(loss).backward()
        scaler_full.unscale_(opt_full)

        torch.nn.utils.clip_grad_norm_(
            full_model.parameters(), 1.0
        )

        scaler_full.step(opt_full)
        scaler_full.update()

        total += loss.item() * images.size(0)
        count += images.size(0)

    epoch_loss = total / count
    full_history.append(epoch_loss)

    print(
        f"SA+RA {epoch+1}/{FULL_EPOCHS}: "
        f"MSE={epoch_loss:.6f}, "
        f"time={time.time()-t0:.1f}s"
    )


# Evaluation

We evaluate PSNR over all combinations of:

- SNR = 1, 4, 7, 10, 13 dB
- C = 32, 64, 96, 128, 192

The official high-resolution SwinJSCC CBR is:

\[
CBR=rac{C}{2\cdot3\cdot2^{2\cdot4}}
=rac{C}{1536}.
\]

This is the correct communication-rate quantity for the original SwinJSCC model.

Do **not** interpret the FP32 PyTorch tensor memory as the paper's transmitted bitrate.


In [ ]:
def psnr(mse):
    return 10 * math.log10(
        1.0 / max(mse, 1e-12)
    )


@torch.no_grad()
def evaluate(model, loader, snr, rate):
    model.eval()

    total_sq_error = 0.0
    total_pixels = 0

    for images in loader:
        images = images.to(
            DEVICE,
            non_blocking=True
        )

        recon, _, _ = model(
            images, snr, rate
        )

        total_sq_error += F.mse_loss(
            recon,
            images,
            reduction="sum"
        ).item()

        total_pixels += images.numel()

    mse = total_sq_error / total_pixels

    return psnr(mse)


rows = []

for snr in TRAIN_SNRS:
    for rate in TRAIN_RATES:

        score = evaluate(
            full_model,
            test_loader,
            snr,
            rate
        )

        cbr = rate / 1536.0

        rows.append({
            "SNR_dB": snr,
            "C": rate,
            "CBR": cbr,
            "PSNR_dB": score
        })

results = pd.DataFrame(rows)

display(results)


In [ ]:
# PSNR heatmap

pivot = results.pivot(
    index="SNR_dB",
    columns="C",
    values="PSNR_dB"
)

plt.figure(figsize=(8, 5))
plt.imshow(pivot.values, aspect="auto")

plt.xticks(
    range(len(pivot.columns)),
    pivot.columns
)

plt.yticks(
    range(len(pivot.index)),
    pivot.index
)

plt.xlabel("C / active channels")
plt.ylabel("SNR (dB)")
plt.title("SwinJSCC SA+RA — PSNR")
plt.colorbar(label="PSNR (dB)")
plt.show()


In [ ]:
# PSNR versus CBR

plt.figure(figsize=(9, 6))

for snr in TRAIN_SNRS:
    d = results[
        results.SNR_dB == snr
    ].sort_values("CBR")

    plt.plot(
        d.CBR,
        d.PSNR_dB,
        marker="o",
        label=f"SNR={snr} dB"
    )

plt.xlabel("CBR")
plt.ylabel("PSNR (dB)")
plt.title("SwinJSCC Rate–Distortion Performance")
plt.grid(True)
plt.legend()
plt.show()


# Compression / representation accounting

For a 256×256 RGB image:

\[
256\cdot256\cdot3\cdot8
=1,572,864	ext{ bits}
=192	ext{ KiB}.
\]

For the full SA+RA bottleneck:

\[
16\cdot16\cdot320
=81,920
\]

FP32 values.

Therefore the raw FP32 continuous latent occupies:

\[
81,920\cdot32
=2,621,440	ext{ bits}
=320	ext{ KiB}.
\]

Again, this **320 KiB is a memory representation**, not the SwinJSCC communication bitrate.

At a selected rate \(C\), the active latent contains:

\[
16\cdot16\cdot C
\]

real-valued channel symbols.

The official CBR remains:

\[
CBR=C/1536.
\]


In [ ]:
# Measure one real latent produced by the trained model.

full_model.eval()

sample = next(iter(test_loader))[:1].to(DEVICE)

with torch.no_grad():
    recon, latent, mask = full_model(
        sample,
        10,
        96
    )

B, N, latent_C = latent.shape

original_bits = (
    IMAGE_SIZE * IMAGE_SIZE * 3 * 8
)

continuous_bits = (
    N * latent_C * 32
)

selected_C = 96

active_fp32_bits = (
    N * selected_C * 32
)

official_cbr = selected_C / 1536.0

print("=" * 80)
print("COMPRESSION / LATENT SIZE REPORT")
print("=" * 80)

print(
    f"Original RGB:              "
    f"{original_bits:,} bits "
    f"({original_bits/8/1024:.2f} KiB)"
)

print(
    f"Continuous latent shape:   "
    f"{tuple(latent.shape)}"
)

print(
    f"Continuous FP32 latent:    "
    f"{continuous_bits:,} bits "
    f"({continuous_bits/8/1024:.2f} KiB)"
)

print(
    f"Selected C:                "
    f"{selected_C}"
)

print(
    f"Active FP32 latent:        "
    f"{active_fp32_bits:,} bits "
    f"({active_fp32_bits/8/1024:.2f} KiB)"
)

print(
    f"Official SwinJSCC CBR:     "
    f"{official_cbr:.6f}"
)

print(
    f"Channel symbols/image:     "
    f"{N * selected_C:,}"
)

print("=" * 80)
print(
    "Important: FP32 tensor size != communication bitrate."
)
print(
    "For SwinJSCC, CBR is the appropriate rate measure."
)
print("=" * 80)


In [ ]:
# Rate table

rate_table = []

for C in TRAIN_RATES:
    cbr = C / 1536.0
    latent_symbols = N * C
    fp32_bits = latent_symbols * 32

    rate_table.append({
        "C": C,
        "CBR": cbr,
        "Channel symbols/image": latent_symbols,
        "Active FP32 latent KiB": fp32_bits / 8 / 1024
    })

rate_table = pd.DataFrame(rate_table)

display(rate_table)


# Visual reconstruction


In [ ]:
@torch.no_grad()
def show_reconstruction(model, images, snr, rate):
    model.eval()

    recon, _, _ = model(
        images.to(DEVICE),
        snr,
        rate
    )

    recon = recon.cpu()

    fig, axes = plt.subplots(
        2, len(images),
        figsize=(4 * len(images), 6)
    )

    for i in range(len(images)):
        axes[0, i].imshow(
            images[i].permute(1,2,0).numpy()
        )
        axes[0, i].set_title("Original")
        axes[0, i].axis("off")

        axes[1, i].imshow(
            recon[i].permute(1,2,0).numpy()
        )
        axes[1, i].set_title(
            f"SNR={snr}, C={rate}"
        )
        axes[1, i].axis("off")

    plt.tight_layout()
    plt.show()


test_images = next(iter(test_loader))[:4]

show_reconstruction(
    full_model,
    test_images,
    10,
    96
)


# 2×T4 verification

The model is wrapped with:

```python
nn.DataParallel(model, device_ids=[0, 1])
```

A sufficiently large batch is split between the two T4 GPUs.

Use the following cell after training to confirm that both GPUs have allocated memory. For live utilization, run `!nvidia-smi` while a training epoch is running.


In [ ]:
print("CUDA GPUs:", torch.cuda.device_count())

if isinstance(full_model, nn.DataParallel):
    print("DataParallel:", True)
    print("Devices:", full_model.device_ids)
    print("Output device:", full_model.output_device)
else:
    print("DataParallel:", False)

for i in range(torch.cuda.device_count()):
    print(
        f"GPU {i}: "
        f"allocated={torch.cuda.memory_allocated(i)/1024**3:.2f} GB, "
        f"reserved={torch.cuda.memory_reserved(i)/1024**3:.2f} GB"
    )

print("\nFor live utilization:")
print("!nvidia-smi")


In [ ]:
# Save trained model and evaluation results.

EXPORT_DIR = Path(
    "/kaggle/working/swinjscc_results"
)
EXPORT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

model_to_save = (
    full_model.module
    if isinstance(full_model, nn.DataParallel)
    else full_model
)

torch.save(
    {
        "model_state_dict": model_to_save.state_dict(),
        "model_name": MODEL_FULL,
        "image_size": IMAGE_SIZE,
        "embed_dims": EMBED_DIMS,
        "depths": DEPTHS,
        "num_heads": NUM_HEADS,
        "train_snrs": TRAIN_SNRS,
        "train_rates": TRAIN_RATES,
        "dataset": str(TRAIN_DIR),
    },
    EXPORT_DIR / "swinjscc_sa_ra_base.pt"
)

results.to_csv(
    EXPORT_DIR / "evaluation_results.csv",
    index=False
)

rate_table.to_csv(
    EXPORT_DIR / "compression_table.csv",
    index=False
)

print("Saved to:", EXPORT_DIR)
